# Preprocesamiento

## 1. Importar paquetes

In [5]:
import pandas as pd
from pathlib import Path

## 2. Carga de los datos

In [7]:
ruta = Path.cwd().resolve()
datos = 'trabajo_resultado_calidad.pickle'

ruta_completa = None
for base in [ruta, *ruta.parents]:
    candidata = base / '02_Datos' / '03_Trabajo' / datos
    if candidata.exists():
        ruta_completa = candidata
        break

if ruta_completa is None:
    raise FileNotFoundError(
        "No se encontro trabajo_resultado_calidad.pickle en 02_Datos/03_Trabajo desde el directorio actual ni sus padres."
    )

repo_root = ruta_completa.parents[2]
df = pd.read_pickle(ruta_completa)

print(f'Datos cargados desde: {ruta_completa}')
print(f'Forma del dataset: {df.shape}')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Github\\Asteroid_Classification\\03_Notebooks\\02_Datos\\03_Trabajo\\trabajo_resultado_calidad.pickle'

## 3. Validacion inicial de variables

Este primer paso valida consistencia basica del dataset antes de homogeneizar variables: tipos, nulos, cardinalidad y dispersion inicial.

In [3]:
# 3.1 Tipos de datos y resumen general
resumen_tipos = (
    df.dtypes.astype(str)
    .rename('dtype')
    .to_frame()
)

resumen_tipos['nulos'] = df.isna().sum()
resumen_tipos['pct_nulos'] = (resumen_tipos['nulos'] / len(df) * 100).round(3)
resumen_tipos['n_unicos'] = df.nunique(dropna=False)

print(f'Registros: {len(df):,} | Variables: {df.shape[1]}')
resumen_tipos.sort_values(['dtype', 'pct_nulos'], ascending=[True, False]).head(20)

Registros: 88,292 | Variables: 35


,dtype,nulos,pct_nulos,n_unicos
H,float64,0,0.0,169
diameter,float64,0,0.0,15124
albedo,float64,0,0.0,955
diameter_sigma,float64,0,0.0,2795
e,float64,0,0.0,88292
a,float64,0,0.0,88292
q,float64,0,0.0,88292
i,float64,0,0.0,88292
om,float64,0,0.0,88292
w,float64,0,0.0,88292


In [4]:
# 3.2 Variables con posibles problemas de consistencia
objetos = resumen_tipos[resumen_tipos['dtype'] == 'object']
altos_nulos = resumen_tipos[resumen_tipos['pct_nulos'] > 0]

print('Variables tipo object:')
display(objetos)

print('Variables con nulos:')
display(altos_nulos.sort_values('pct_nulos', ascending=False))

Variables tipo object:


,dtype,nulos,pct_nulos,n_unicos
full_name,object,0,0.0,88292
pha,object,0,0.0,2
class,object,0,0.0,11


Variables con nulos:


,dtype,nulos,pct_nulos,n_unicos


In [5]:
# 3.3 Dispersion inicial de variables numericas (base para homogeneizacion)
num_cols = df.select_dtypes(include=['number']).columns

resumen_numericas = df[num_cols].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
resumen_numericas['rango'] = resumen_numericas['max'] - resumen_numericas['min']
resumen_numericas['iqr'] = resumen_numericas['75%'] - resumen_numericas['25%']

# Coeficiente de variacion aproximado para medir heterogeneidad de escala
resumen_numericas['cv_pct'] = (
    (resumen_numericas['std'] / resumen_numericas['mean'].replace(0, pd.NA)) * 100
).round(2)

resumen_numericas[['mean', 'std', 'min', '1%', '25%', '50%', '75%', '99%', 'max', 'rango', 'iqr', 'cv_pct']].sort_values('cv_pct', ascending=False).head(20)

,mean,std,min,1%,25%,50%,75%,99%,max,rango,iqr,cv_pct
sigma_per,1.451289e-03,3.213364e-01,9.439100e-09,4.997266e-06,1.180500e-05,1.896300e-05,3.055450e-05,1.974853e-04,9.474600e+01,9.474600e+01,1.874950e-05,22141.45
sigma_ad,6.148522e-07,1.219626e-04,1.115900e-11,7.034148e-09,1.541075e-08,2.342700e-08,3.656100e-08,1.957527e-07,3.574800e-02,3.574800e-02,2.115025e-08,19836.08
sigma_a,3.877699e-07,7.257304e-05,1.035600e-11,6.162600e-09,1.339875e-08,2.056650e-08,3.225300e-08,1.738835e-07,2.123000e-02,2.123000e-02,1.885425e-08,18715.49
sigma_tp,2.545417e-04,1.496514e-02,3.686700e-08,3.028364e-05,7.292775e-05,1.148300e-04,1.981125e-04,1.454100e-03,4.440300e+00,4.440300e+00,1.251848e-04,5879.25
sigma_q,1.958960e-07,3.306094e-06,1.947000e-11,6.827619e-08,1.105600e-07,1.418500e-07,1.905200e-07,7.255834e-07,9.040900e-04,9.040900e-04,7.996000e-08,1687.68
sigma_e,6.362810e-08,3.714860e-07,4.807000e-12,2.926400e-08,4.086100e-08,4.983100e-08,6.448250e-08,2.236245e-07,9.598200e-05,9.598200e-05,2.362150e-08,583.84
sigma_ma,3.888396e-05,1.099415e-04,7.355900e-09,7.670591e-06,1.660900e-05,2.432400e-05,3.958350e-05,2.438972e-04,1.967900e-02,1.967899e-02,2.297450e-05,282.74
sigma_om,5.842985e-05,1.240589e-04,6.176900e-08,1.022491e-05,2.304475e-05,3.502150e-05,6.137075e-05,3.686496e-04,1.144000e-02,1.143994e-02,3.832600e-05,212.32
sigma_n,2.926238e-09,6.058489e-09,1.201400e-12,1.121900e-09,1.762700e-09,2.283850e-09,3.213400e-09,1.065972e-08,8.751600e-07,8.751588e-07,1.450700e-09,207.04
sigma_w,7.818983e-05,1.582236e-04,6.618400e-08,1.713246e-05,3.507775e-05,5.126350e-05,8.274700e-05,4.626656e-04,1.843200e-02,1.843193e-02,4.766925e-05,202.36


## 4. Homogeneizacion de tipos y formatos

En este paso se estandarizan formatos de texto, se convierten columnas numericas al tipo correcto y se normalizan variables categoricas para dejar una base consistente para modelado.

In [ ]:
# 4.1 Estandarizar formatos de texto
if 'df' not in globals():
    raise NameError("Primero ejecuta la celda '2. Carga de los datos' para definir df.")

df_h = df.copy()
dtypes_antes = df_h.dtypes.astype(str)

# Normalizacion de espacios y tokens de nulo en columnas tipo texto
obj_cols = df_h.select_dtypes(include=['object']).columns
for col in obj_cols:
    df_h[col] = (
        df_h[col]
        .astype(str)
        .str.strip()
        .replace({'?': pd.NA, '': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )

# Ajustes especificos de formato
if 'full_name' in df_h.columns:
    df_h['full_name'] = df_h['full_name'].str.replace("'", '', regex=False).str.strip()

if 'pha' in df_h.columns:
    df_h['pha'] = df_h['pha'].str.upper().str.strip()

if 'class' in df_h.columns:
    df_h['class'] = df_h['class'].str.upper().str.strip()

print('Columnas de texto normalizadas:', len(obj_cols))

NameError: name 'df' is not defined

In [ ]:
# 4.2 Homogeneizar columnas numericas
# Columnas esperadas como numericas segun el diccionario del proyecto
numeric_cols = [
    'spkid', 'H', 'diameter', 'albedo', 'diameter_sigma',
    'e', 'a', 'q', 'i', 'om', 'w', 'ma', 'ad', 'n', 'tp', 'tp_cal',
    'per', 'per_y', 'moid', 'moid_ld',
    'sigma_e', 'sigma_a', 'sigma_q', 'sigma_i', 'sigma_om', 'sigma_w',
    'sigma_ma', 'sigma_ad', 'sigma_n', 'sigma_tp', 'sigma_per', 'rms'
]

numeric_cols = [c for c in numeric_cols if c in df_h.columns]

for col in numeric_cols:
    df_h[col] = pd.to_numeric(df_h[col], errors='coerce')

# spkid se mantiene como entero nullable para preservar faltantes si aparecen
if 'spkid' in df_h.columns:
    df_h['spkid'] = df_h['spkid'].round().astype('Int64')

print('Columnas numericas convertidas:', len(numeric_cols))

In [ ]:
# 4.3 Convertir categoricas a tipo category
cat_cols = [c for c in ['pha', 'class'] if c in df_h.columns]
for col in cat_cols:
    df_h[col] = df_h[col].astype('category')

print('Columnas categoricas convertidas:', cat_cols)
df_h[cat_cols].dtypes if cat_cols else 'No hay columnas categoricas objetivo.'

In [ ]:
# 4.4 Resumen de homogeneizacion
resumen_dtypes = pd.DataFrame({
    'dtype_antes': dtypes_antes,
    'dtype_despues': df_h.dtypes.astype(str)
})
resumen_dtypes['cambio_tipo'] = resumen_dtypes['dtype_antes'] != resumen_dtypes['dtype_despues']

resumen_nulos = pd.DataFrame({
    'nulos_despues': df_h.isna().sum(),
    'pct_nulos_despues': (df_h.isna().sum() / len(df_h) * 100).round(3)
}).sort_values('pct_nulos_despues', ascending=False)

print('Variables con cambio de tipo:')
display(resumen_dtypes[resumen_dtypes['cambio_tipo']].sort_index())

print('Top variables con nulos despues de homogeneizar:')
display(resumen_nulos.head(15))

In [ ]:
# 4.5 Persistir dataset homogeneizado
ruta_salida = repo_root / '02_Datos' / '03_Trabajo' / 'trabajo_homogeneizado.pickle'
df_h.to_pickle(ruta_salida)

print(f'Dataset homogeneizado guardado en: {ruta_salida}')